# Water Quality Data Cleaning: EPA Measurements

Turns the long EPA/WQX **results** table (~971K rows, one row per measurement)
into a tidy wide table with **one row per station-date** and **one column per
parameter**, ready to merge with station metadata and climate features.

**Input:**  `data/tabular/01_raw/water-quality/epa-wq.csv`
**Output:** `data/tabular/02_clean/water-quality/epa-wq-clean.csv`

**Pipeline**
1. Load only the columns we need.
2. **Filter rows** — keep configured parameters, drop QC samples (blanks /
   replicates / duplicates), keep `Actual`/`Calculated` results, coerce the
   measurement to numeric.
3. **Standardize units** per parameter so every value in a column shares one
   unit — including molar conversion of nutrient speciation
   (e.g. nitrate-as-NO3 -> nitrate-as-N). Incompatible units (loads, areal,
   percent-saturation) are dropped.
4. **Range-validate** each parameter against physical bounds (nulls impossible
   values such as pH 805 or DO 581 mg/L).
5. **De-duplicate** replicate measurements with the median per station-date-parameter.
6. **Pivot** to wide and save.

**Why these changes matter** — the previous version selected the 30 most frequent
characteristics (mixing in habitat/observation fields), converted only
temperature, never reconciled the other units, and applied no range checks, so a
single column like `Nitrate_value` blended `mg/L as N`, `mg/L as NO3` and `g/m2`,
and impossible values flowed straight through to the models.

> **Path note:** targets the migrated `01_raw`/`02_clean` layout. The merge
> notebook and README still point at `data/tabular/water-quality/{raw,clean}`
> and should be updated to read from `02_clean/water-quality/`.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    """Walk upward until we find the repo's data/tabular directory.

    Notebooks have no __file__, and the kernel's working directory varies
    (repo root vs. the notebook folder), so resolving paths relative to a
    fixed number of "../" is fragile. Searching upward for a sentinel makes
    the notebook runnable from anywhere.
    """
    here = (start or Path.cwd()).resolve()
    for candidate in (here, *here.parents):
        if (candidate / "data" / "tabular").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate repo root containing data/tabular/")


REPO_ROOT = find_repo_root()
RAW_DIR = REPO_ROOT / "data" / "tabular" / "01_raw" / "water-quality"
CLEAN_DIR = REPO_ROOT / "data" / "tabular" / "02_clean" / "water-quality"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)
print("Repo root:", REPO_ROOT)
print("Raw dir:  ", RAW_DIR)
print("Clean dir:", CLEAN_DIR)

Repo root: /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN Fellowship Project 2026/workspace/Water-Quality-Prediction
Raw dir:   /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN Fellowship Project 2026/workspace/Water-Quality-Prediction/data/tabular/01_raw/water-quality
Clean dir: /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN Fellowship Project 2026/workspace/Water-Quality-Prediction/data/tabular/02_clean/water-quality


## Step 1 — Load (only the columns we need)

The raw file has 81 columns; we need 11. Reading a subset (and setting
`low_memory=False`) is faster and silences the mixed-dtype warning. We
deliberately **drop the inline `ActivityLocation` lat/lon** — those are corrupt
in the raw data (values like lat `-1.0`, lon `+92.5`); coordinates come from the
cleaned stations table via the join instead.

In [2]:
USECOLS = [
    "ActivityIdentifier",
    "ActivityStartDate",
    "ActivityStartTime/Time",
    "MonitoringLocationIdentifier",
    "CharacteristicName",
    "ResultMeasureValue",
    "ResultMeasure/MeasureUnitCode",
    "ActivityTypeCode",
    "ResultValueTypeName",
    "ResultDetectionConditionText",
    "MeasureQualifierCode",
]
df = pd.read_csv(RAW_DIR / "epa-wq.csv", usecols=USECOLS, low_memory=False)
print(f"Loaded {len(df):,} measurement rows")
df.head()

Loaded 970,946 measurement rows


,ActivityIdentifier,ActivityTypeCode,ActivityStartDate,ActivityStartTime/Time,MonitoringLocationIdentifier,ResultDetectionConditionText,CharacteristicName,ResultMeasureValue,ResultMeasure/MeasureUnitCode,MeasureQualifierCode,ResultValueTypeName
0,nwisia.01.01600010,Sample-Routine,2015-10-14,10:02:00,USGS-05465500,NaN,"Stream flow, instantaneous",5290,ft3/s,NaN,Actual
1,nwisia.01.01600010,Sample-Routine,2015-10-14,10:02:00,USGS-05465500,NaN,Number of sampling points,1,count,NaN,Actual
2,nwisia.01.01600010,Sample-Routine,2015-10-14,10:02:00,USGS-05465500,NaN,"Height, gage",12.18,ft,NaN,Actual
3,nwisia.01.01600010,Sample-Routine,2015-10-14,10:02:00,USGS-05465500,NaN,"Height, gage",3.71,m,NaN,Actual
4,nwisia.01.01600010,Sample-Routine,2015-10-14,10:02:00,USGS-05465500,NaN,"Stream flow, instantaneous",150,m3/sec,NaN,Actual


## Step 2 — Parameter configuration

All domain knowledge — which parameters to keep, their canonical units, the unit conversions, and the valid ranges — lives in one declarative dictionary so the pipeline is transparent and easy to extend.

In [3]:
# Molar-mass conversion factors for nutrient speciation (mg of compound -> mg of N or P).
NO3_TO_N = 14.007 / 62.004   # nitrate as NO3 -> as N
NO2_TO_N = 14.007 / 46.005   # nitrite as NO2 -> as N
PO4_TO_P = 30.974 / 94.971   # orthophosphate as PO4 -> as P

# For every parameter we keep:
#   canonical_unit : the single unit every value in the output column is expressed in
#   conversions    : {raw_unit: multiplicative_factor} -> any raw unit NOT listed is
#                    considered incompatible and those rows are dropped (e.g. tons/day
#                    loads, areal g/m2, percent-saturation DO). "" matches a missing unit.
#   valid_range    : (low, high) physically plausible bounds; values outside are nulled
# Bare "mg/L" nitrogen/phosphorus values are assumed to follow the USGS/EPA convention
# of being reported as N / as P respectively.
PARAMETERS = {
    "Temperature, water": {
        "canonical_unit": "deg C",
        "conversions": {"deg C": 1.0},   # deg F handled affinely before this map
        "valid_range": (-1.0, 40.0),
    },
    "Dissolved oxygen (DO)": {
        "canonical_unit": "mg/L",
        "conversions": {"mg/L": 1.0, "ppm": 1.0},
        "valid_range": (0.0, 25.0),
    },
    "pH": {
        "canonical_unit": "std units",
        "conversions": {"": 1.0, "std units": 1.0},
        "valid_range": (2.0, 12.0),
    },
    "Nitrate": {
        "canonical_unit": "mg/L as N",
        "conversions": {"mg/L": 1.0, "mg/l": 1.0, "mg/l as N": 1.0,
                        "mg N/l******": 1.0, "mg/l asNO3": NO3_TO_N},
        "valid_range": (0.0, 150.0),
    },
    "Nitrite": {
        "canonical_unit": "mg/L as N",
        "conversions": {"mg/L": 1.0, "mg/l": 1.0, "mg/l as N": 1.0,
                        "mg N/l******": 1.0, "mg/l asNO2": NO2_TO_N},
        "valid_range": (0.0, 10.0),
    },
    "Nitrate + Nitrite": {
        "canonical_unit": "mg/L as N",
        "conversions": {"mg/L": 1.0, "mg/l": 1.0},
        "valid_range": (0.0, 100.0),
    },
    "Ammonia-nitrogen": {
        "canonical_unit": "mg/L as N",
        "conversions": {"mg/L": 1.0, "mg/l": 1.0, "ug/L": 0.001, "mg N/l******": 1.0},
        "valid_range": (0.0, 50.0),
    },
    "Kjeldahl nitrogen": {
        "canonical_unit": "mg/L",
        "conversions": {"mg/L": 1.0, "mg/l": 1.0, "mg/l as N": 1.0},
        "valid_range": (0.0, 100.0),
    },
    "Orthophosphate": {
        "canonical_unit": "mg/L as P",
        "conversions": {"mg/L": 1.0, "mg/l": 1.0, "mg/l as P": 1.0,
                        "mg/l asPO4": PO4_TO_P, "ug/L": 0.001},
        "valid_range": (0.0, 10.0),
    },
    "Phosphate-phosphorus": {
        "canonical_unit": "mg/L as P",
        "conversions": {"mg/L": 1.0, "mg/l": 1.0, "ug/L": 0.001},
        "valid_range": (0.0, 10.0),
    },
    "Total Phosphorus, mixed forms": {
        "canonical_unit": "mg/L as P",
        "conversions": {"mg/L": 1.0, "mg/l": 1.0, "ug/L": 0.001},
        "valid_range": (0.0, 10.0),
    },
    "Chloride": {
        "canonical_unit": "mg/L",
        "conversions": {"mg/L": 1.0, "mg/l": 1.0},
        "valid_range": (0.0, 5000.0),
    },
    "Sulfate": {
        "canonical_unit": "mg/L",
        "conversions": {"mg/L": 1.0, "mg/l": 1.0},
        "valid_range": (0.0, 2000.0),
    },
    "Specific conductance": {
        "canonical_unit": "uS/cm",
        "conversions": {"uS/cm": 1.0, "uS/cm @25C": 1.0, "umho/cm": 1.0},
        "valid_range": (0.0, 10000.0),
    },
    "Total dissolved solids": {
        "canonical_unit": "mg/L",
        "conversions": {"mg/L": 1.0, "mg/l": 1.0},   # tons/ac ft, tons/day dropped
        "valid_range": (0.0, 50000.0),
    },
    "Total suspended solids": {
        "canonical_unit": "mg/L",
        "conversions": {"mg/L": 1.0, "mg/l": 1.0},
        "valid_range": (0.0, 50000.0),
    },
    "Turbidity": {
        "canonical_unit": "NTU",   # NTU/FNU/NTRU/FBRU/FNMU treated as comparable
        "conversions": {"NTU": 1.0, "FNU": 1.0, "NTRU": 1.0, "FBRU": 1.0, "FNMU": 1.0},
        "valid_range": (0.0, 5000.0),
    },
    "Escherichia coli": {
        "canonical_unit": "MPN/100mL",
        "conversions": {"MPN/100mL": 1.0, "MPN/100 ml": 1.0, "#/100mL": 1.0,
                        "MPN": 1.0, "cfu/100mL": 1.0},
        "valid_range": (0.0, 3_000_000.0),
    },
    "Chlorophyll a, free of pheophytin": {
        "canonical_unit": "ug/L",
        "conversions": {"ug/L": 1.0, "mg/L": 1000.0},   # ug/cm2 (periphyton) dropped
        "valid_range": (0.0, 2000.0),
    },
}
print(f"{len(PARAMETERS)} parameters configured.")

19 parameters configured.


## Step 3 — Filter rows

Each filter prints how many rows it removes, so the pipeline is auditable.

- **Parameters:** keep only the configured characteristics.
- **QC samples:** drop `Quality Control ...` activities (field blanks, replicates,
  lab duplicates). These were previously impossible to remove because
  `ActivityTypeCode` was dropped before this point.
- **Result type:** keep `Actual` and `Calculated`; drop `Estimated`.
- **Numeric:** coerce the value; non-numeric entries are text observations
  ("sunny", "Moderate") or non-detects, which we report and drop.

In [4]:
def step(df_in, mask, label):
    kept = df_in[mask]
    print(f"{label:<42} {len(df_in):>9,} -> {len(kept):>9,}  (-{len(df_in)-len(kept):,})")
    return kept

n0 = len(df)
df = step(df, df["CharacteristicName"].isin(PARAMETERS), "keep configured parameters")
df = step(df, ~df["ActivityTypeCode"].astype("string").str.startswith("Quality Control", na=False),
          "drop QC samples")
df = step(df, df["ResultValueTypeName"].isin(["Actual", "Calculated"]), "keep Actual/Calculated")

# Report censored / non-detect rows before they are dropped by numeric coercion.
value_num = pd.to_numeric(df["ResultMeasureValue"], errors="coerce")
non_numeric = df["ResultMeasureValue"].notna() & value_num.isna()
nd = df.loc[non_numeric, "ResultDetectionConditionText"].notna().sum()
print(f"\nNon-numeric values dropped: {int(non_numeric.sum()):,} "
      f"(of which flagged non-detect/censored: {int(nd):,})")
df = df.assign(value=value_num)
df = step(df, df["value"].notna() & df["MonitoringLocationIdentifier"].notna(),
          "drop missing value / station id")
print(f"\nRetained {len(df):,} of {n0:,} raw rows ({len(df)/n0:.1%}).")

keep configured parameters                   970,946 ->   362,723  (-608,223)
drop QC samples                              362,723 ->   360,656  (-2,067)
keep Actual/Calculated                       360,656 ->   360,384  (-272)

Non-numeric values dropped: 402 (of which flagged non-detect/censored: 27)
drop missing value / station id              360,384 ->   325,863  (-34,521)

Retained 325,863 of 970,946 raw rows (33.6%).


## Step 4 — Standardize units

For every (parameter, raw unit) pair we look up a multiplicative factor from the
config. Temperature in `deg F` is converted affinely first, then relabeled to
`deg C` so it flows through the multiplicative map. Rows whose unit is **not**
listed for that parameter are incompatible and dropped.

In [5]:
df["unit"] = df["ResultMeasure/MeasureUnitCode"].astype("string").str.strip().fillna("")

# Affine pre-pass: Fahrenheit -> Celsius, then relabel so the factor map applies.
is_f = (df["CharacteristicName"] == "Temperature, water") & (df["unit"] == "deg F")
df.loc[is_f, "value"] = (df.loc[is_f, "value"] - 32.0) * 5.0 / 9.0
df.loc[is_f, "unit"] = "deg C"
print(f"Temperature readings converted deg F -> deg C: {int(is_f.sum()):,}")

# Build a (parameter, raw_unit) -> factor / canonical_unit lookup and merge it in.
conv_rows = [
    {"CharacteristicName": p, "unit": u, "factor": f, "canonical_unit": spec["canonical_unit"]}
    for p, spec in PARAMETERS.items() for u, f in spec["conversions"].items()
]
conv = pd.DataFrame(conv_rows)

before = len(df)
df = df.merge(conv, on=["CharacteristicName", "unit"], how="left")
incompatible = df["factor"].isna()
print(f"Rows with incompatible/unconvertible units dropped: {int(incompatible.sum()):,}")
df = df[~incompatible].copy()
df["value"] = df["value"] * df["factor"]
print(f"Standardized {len(df):,} rows (was {before:,}).")

Temperature readings converted deg F -> deg C: 15,127
Rows with incompatible/unconvertible units dropped: 2,314
Standardized 323,549 rows (was 325,863).


## Step 5 — Range validation

Null out physically impossible values (e.g. pH outside 2–12, negative DO), then drop the rows that fail. This is what stops garbage like the raw `pH = 805` or `Nitrate = -348` from reaching the models.

In [6]:
lo = df["CharacteristicName"].map(lambda p: PARAMETERS[p]["valid_range"][0])
hi = df["CharacteristicName"].map(lambda p: PARAMETERS[p]["valid_range"][1])
in_range = df["value"].between(lo, hi)
print("Out-of-range values dropped, by parameter:")
print(df.loc[~in_range, "CharacteristicName"].value_counts().to_string())
df = df[in_range].copy()
print(f"\nRows after range validation: {len(df):,}")

Out-of-range values dropped, by parameter:
CharacteristicName
Temperature, water                   278
Orthophosphate                        30
Turbidity                             27
Phosphate-phosphorus                  25
pH                                    17
Dissolved oxygen (DO)                 16
Nitrate                                8
Chlorophyll a, free of pheophytin      5
Kjeldahl nitrogen                      5
Ammonia-nitrogen                       1

Rows after range validation: 323,137


## Step 6 — Build the sample timestamp

In [7]:
dt = df["ActivityStartDate"].astype("string").str.cat(
    df["ActivityStartTime/Time"].astype("string"), sep=" ", na_rep="")
df["ActivityStartDateTime"] = pd.to_datetime(dt.str.strip(), errors="coerce")
missing_dt = df["ActivityStartDateTime"].isna().sum()
print(f"Rows dropped for unparseable date/time: {int(missing_dt):,}")
df = df.dropna(subset=["ActivityStartDateTime"]).copy()

Rows dropped for unparseable date/time: 3,593


## Step 7 — De-duplicate, then pivot to wide

Several measurements of the same parameter can share a station and timestamp
(profiles, re-reads). We collapse them with the **median** — robust to a stray
bad replicate — rather than arbitrarily taking the first. The result is pivoted
so each parameter becomes a `<parameter>_value` column, with a constant
`<parameter>_unit` companion documenting the canonical unit.

In [8]:
agg = (df.groupby(["MonitoringLocationIdentifier", "ActivityStartDateTime",
                   "CharacteristicName"], as_index=False)["value"].median())

wide = agg.pivot(index=["MonitoringLocationIdentifier", "ActivityStartDateTime"],
                 columns="CharacteristicName", values="value")
wide.columns = [f"{c}_value" for c in wide.columns]
wide = wide.reset_index()

# Add a constant unit column next to each value column (canonical unit per parameter).
for param, spec in PARAMETERS.items():
    vcol = f"{param}_value"
    if vcol in wide.columns:
        wide[f"{param}_unit"] = wide[vcol].notna().map({True: spec["canonical_unit"], False: pd.NA})

# Order columns: keys first, then value/unit pairs grouped per parameter.
ordered = ["MonitoringLocationIdentifier", "ActivityStartDateTime"]
for param in PARAMETERS:
    ordered += [c for c in (f"{param}_value", f"{param}_unit") if c in wide.columns]
df_samples = wide[ordered]
print(f"Wide table: {df_samples.shape[0]:,} station-date rows x {df_samples.shape[1]} columns")
df_samples.head()

Wide table: 47,823 station-date rows x 40 columns


,MonitoringLocationIdentifier,ActivityStartDateTime,"Temperature, water_value","Temperature, water_unit",Dissolved oxygen (DO)_value,Dissolved oxygen (DO)_unit,pH_value,pH_unit,Nitrate_value,Nitrate_unit,...,Total dissolved solids_value,Total dissolved solids_unit,Total suspended solids_value,Total suspended solids_unit,Turbidity_value,Turbidity_unit,Escherichia coli_value,Escherichia coli_unit,"Chlorophyll a, free of pheophytin_value","Chlorophyll a, free of pheophytin_unit"
0,11NPSWRD_WQX-HTLN_EFMO_DOUS1,2017-07-18 15:00:00,19.820,deg C,8.75,mg/L,8.33000,std units,NaN,NaN,...,NaN,NaN,NaN,NaN,4.60000,NTU,NaN,NaN,NaN,NaN
1,11NPSWRD_WQX-HTLN_EFMO_DOUS1,2023-07-19 11:00:00,16.372,deg C,12.24,mg/L,8.21500,std units,NaN,NaN,...,NaN,NaN,NaN,NaN,0.94000,NTU,NaN,NaN,NaN,NaN
2,11NPSWRD_WQX-HTLN_HEHO_HOOV1,2017-07-17 17:00:00,19.920,deg C,7.62,mg/L,8.19500,std units,NaN,NaN,...,NaN,NaN,NaN,NaN,17.46000,NTU,NaN,NaN,NaN,NaN
3,11NPSWRD_WQX-HTLN_HEHO_HOOV1,2022-07-05 16:00:00,17.569,deg C,6.71,mg/L,7.74348,std units,NaN,NaN,...,NaN,NaN,NaN,NaN,20.11272,NTU,NaN,NaN,NaN,NaN
4,11NPSWRD_WQX-HTLN_HEHO_HOOV1,2023-07-17 15:00:00,16.622,deg C,6.75,mg/L,7.93048,std units,NaN,NaN,...,NaN,NaN,NaN,NaN,3.90000,NTU,NaN,NaN,NaN,NaN


## Step 8 — Sanity check

Confirm the modeling targets are present and now sit inside physically sensible ranges.

In [9]:
targets = ["Temperature, water_value", "pH_value",
           "Dissolved oxygen (DO)_value", "Nitrate_value"]
df_samples[targets].describe().round(2)

,"Temperature, water_value",pH_value,Dissolved oxygen (DO)_value,Nitrate_value
count,34705.00,32353.00,31825.00,12348.00
mean,16.12,7.99,9.12,3.34
std,8.31,0.66,2.67,5.32
min,-1.00,4.00,0.00,0.00
25%,10.00,7.80,7.90,0.00
50%,18.00,8.00,8.81,2.00
75%,23.00,8.30,10.51,5.00
max,37.00,11.76,24.60,132.32


## Step 9 — Save

In [10]:
out_file = CLEAN_DIR / "epa-wq-clean.csv"
df_samples.to_csv(out_file, index=False)
print(f"Saved {len(df_samples):,} station-date rows -> {out_file}")
print(f"Parameters in output: {sum(c.endswith('_value') for c in df_samples.columns)}")

Saved 47,823 station-date rows -> /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN Fellowship Project 2026/workspace/Water-Quality-Prediction/data/tabular/02_clean/water-quality/epa-wq-clean.csv
Parameters in output: 19
